# Number of Children Under Age 1 in Michigan

Calculates the weighted count of persons with `age < 1` in Michigan.

**Two approaches compared:**
1. State-specific dataset (`MI.h5`)
2. National dataset filtered to Michigan via `state_code`

Result: the state dataset significantly undercounts MI population (~4.1M vs actual ~10M), so the national dataset gives the realistic figure.

In [1]:
from policyengine_us import Microsimulation
from huggingface_hub import hf_hub_download

YEAR = 2026

## Approach 1: State-specific MI.h5 dataset

In [2]:
mi_path = hf_hub_download(
    repo_id="policyengine/policyengine-us-data",
    filename="states/MI.h5",
    repo_type="model",
)
sim_state = Microsimulation(dataset=mi_path)
age_state = sim_state.calculate("age", YEAR)

# (boolean_series).sum() applies weights once -> weighted count of True rows
state_total = (age_state >= 0).sum()
state_under_1 = (age_state < 1).sum()
print(f"MI total population (state dataset): {state_total:,.0f}")
print(f"MI children under age 1 (state dataset): {state_under_1:,.0f}")

MI total population (state dataset): 4,104,577
MI children under age 1 (state dataset): 43,689


## Approach 2: National dataset filtered to Michigan

In [3]:
sim_nat = Microsimulation()
age_nat = sim_nat.calculate("age", YEAR).values
weight_nat = sim_nat.calculate("person_weight", YEAR).values
state_code_person = sim_nat.calculate("state_code", YEAR, map_to="person").values

in_mi = state_code_person == "MI"
mi_total = float(weight_nat[in_mi].sum())
mi_under_1 = float(weight_nat[(age_nat < 1) & in_mi].sum())
print(f"MI total population (national dataset): {mi_total:,.0f}")
print(f"MI children under age 1 (national dataset): {mi_under_1:,.0f}")

MI total population (national dataset): 10,106,712
MI children under age 1 (national dataset): 101,162


## Comparison

In [4]:
print(f"{'Source':<35} {'MI total':>15} {'Age < 1':>12}")
print("-" * 64)
print(f"{'State MI.h5 dataset':<35} {state_total:>15,.0f} {state_under_1:>12,.0f}")
print(f"{'National dataset (filter to MI)':<35} {mi_total:>15,.0f} {mi_under_1:>12,.0f}")
print(f"{'Census / MDHHS (real-world)':<35} {'~10,100,000':>15} {'~104,000':>12}")

Source                                     MI total      Age < 1
----------------------------------------------------------------
State MI.h5 dataset                       4,104,577       43,689
National dataset (filter to MI)          10,106,712      101,162
Census / MDHHS (real-world)             ~10,100,000     ~104,000
